<a href="https://colab.research.google.com/github/armanut86/OnlinePort/blob/main/onlinePort.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <img src="https://media.istockphoto.com/id/504862904/photo/container-ship-in-the-harbor-in-asia.jpg?s=612x612&w=0&k=20&c=22ntYAlFmYlRVBBgw3sTiivV4Xx66SPEaa186sMlb-s=" height="100" /> _Port Optimisation Tutorial_

## Instructions
add port of tyne picture
1. Work on a copy of this notebook: _File_ > _Save a copy in Drive_ (you will need a Google account). Alternatively, you can download the notebook using _File_ > _Download .ipynb_, then upload it to [Colab](https://colab.research.google.com/).
2. Execute the following cell (This takes a couple of minutes).
3. Reload this page (press Ctrl+R or the F5 key) and continue to the next section.
4. If the checking the instalation section runs without any errors instalation is succesful.
5. Install the necessary packages for this code by running the package section.
6. Upload the port network before running the final code.
_Notes_:
*

#Installation

In [ ]:
%%shell
set -e

#---------------------------------------------------#
JULIA_VERSION="1.8.2" # any version ≥ 0.7.0
JULIA_PACKAGES="IJulia BenchmarkTools"
JULIA_PACKAGES_IF_GPU="CUDA" # or CuArrays for older Julia versions
JULIA_NUM_THREADS=2
#---------------------------------------------------#

if [ -z `which julia` ]; then
  # Install Julia
  JULIA_VER=`cut -d '.' -f -2 <<< "$JULIA_VERSION"`
  echo "Installing Julia $JULIA_VERSION on the current Colab Runtime..."
  BASE_URL="https://julialang-s3.julialang.org/bin/linux/x64"
  URL="$BASE_URL/$JULIA_VER/julia-$JULIA_VERSION-linux-x86_64.tar.gz"
  wget -nv $URL -O /tmp/julia.tar.gz # -nv means "not verbose"
  tar -x -f /tmp/julia.tar.gz -C /usr/local --strip-components 1
  rm /tmp/julia.tar.gz

  # Install Packages
  nvidia-smi -L &> /dev/null && export GPU=1 || export GPU=0
  if [ $GPU -eq 1 ]; then
    JULIA_PACKAGES="$JULIA_PACKAGES $JULIA_PACKAGES_IF_GPU"
  fi
  for PKG in `echo $JULIA_PACKAGES`; do
    echo "Installing Julia package $PKG..."
    julia -e 'using Pkg; pkg"add '$PKG'; precompile;"' &> /dev/null
  done

  # Install kernel and rename it to "julia"
  echo "Installing IJulia kernel..."
  julia -e 'using IJulia; IJulia.installkernel("julia", env=Dict(
      "JULIA_NUM_THREADS"=>"'"$JULIA_NUM_THREADS"'"))'
  KERNEL_DIR=`julia -e "using IJulia; print(IJulia.kerneldir())"`
  KERNEL_NAME=`ls -d "$KERNEL_DIR"/julia*`
  mv -f $KERNEL_NAME "$KERNEL_DIR"/julia

  echo ''
  echo "Successfully installed `julia -v`!"
  echo "Please reload this page (press Ctrl+R, ⌘+R, or the F5 key) then"
  echo "jump to the 'Checking the Installation' section."
fi

Installing Julia 1.8.2 on the current Colab Runtime...
2023-05-16 11:21:34 URL:https://storage.googleapis.com/julialang2/bin/linux/x64/1.8/julia-1.8.2-linux-x86_64.tar.gz [135859273/135859273] -> "/tmp/julia.tar.gz" [1]
Installing Julia package IJulia...
Installing Julia package BenchmarkTools...
Installing IJulia kernel...
[ Info: Installing julia kernelspec in /root/.local/share/jupyter/kernels/julia-1.8

Please reload this page (press Ctrl+R, ⌘+R, or the F5 key) then
jump to the 'Checking the Installation' section.


## Checking the Installation
The `versioninfo()` function should print your Julia version and some other info about the system:

In [ ]:
versioninfo()

Julia Version 1.8.2
Commit 36034abf260 (2022-09-29 15:21 UTC)
Platform Info:
  OS: Linux (x86_64-linux-gnu)
  CPU: 2 × Intel(R) Xeon(R) CPU @ 2.20GHz
  WORD_SIZE: 64
  LIBM: libopenlibm
  LLVM: libLLVM-13.0.1 (ORCJIT, broadwell)
  Threads: 2 on 2 virtual cores
Environment:
  LD_LIBRARY_PATH = /usr/local/nvidia/lib:/usr/local/nvidia/lib64
  JULIA_NUM_THREADS = 2


## Need Help about Julia?

* Learning: https://julialang.org/learning/
* Documentation: https://docs.julialang.org/
* Questions & Discussions:
  * https://discourse.julialang.org/
  * http://julialang.slack.com/
  * https://stackoverflow.com/questions/tagged/julia


If you need to add new code cells by clicking the `+ Code` button (or _Insert_ > _Code cell_).


<img src="https://www.burohappold.com/wp-content/uploads/2016/04/Newcastle-USB_00_Hawkins-Brown-Kristen-McCluskie.jpg" height="100" />

#Packages

In [ ]:
using     Pkg
          pkg"add JuMP;"
          pkg"add Plots;"
          pkg"add MAT;"
          pkg"add Juniper;"
          pkg"add DelimitedFiles;"
          pkg"add Ipopt;"



    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed IrrationalConstants ── v0.2.2
   Installed DiffRules ──────────── v1.13.0
   Installed DiffResults ────────── v1.1.0
   Installed CodecBzip2 ─────────── v0.7.2
   Installed Bzip2_jll ──────────── v1.0.8+0
   Installed MutableArithmetics ─── v1.3.0
   Installed StaticArraysCore ───── v1.4.0
   Installed SpecialFunctions ───── v2.2.0
   Installed StaticArrays ───────── v1.5.25
   Installed NaNMath ────────────── v1.0.2
   Installed OrderedCollections ─── v1.6.0
   Installed JuMP ───────────────── v1.11.0
   Installed TranscodingStreams ─── v0.9.13
   Installed CommonSubexpressions ─ v0.3.0
   Installed SnoopPrecompile ────── v1.0.3
   Installed LogExpFunctions ────── v0.3.23
   Installed DataStructures ─────── v0.18.13
   Installed ForwardDiff ────────── v0.10.35
   Installed ChainRulesCore ─────── v1.16.0
   Installed OpenSpecFun_jll ────── v0.5.5+0
   Installed Compat ─────────────

# Main code

In [ ]:
using JuMP
using Ipopt
using Plots
using MAT
using Juniper
using DelimitedFiles

function es_dsr(π)

    Δt = delt;
    ΔT = tb-te+1;
    # T = 48;
    tst = tb;
    tfin = te;
    Cen = 0.20;
    Pmax = Pmaxes;
    Emax = 2*Pmax;
    ηc = 0.98;
    ηd = 0.98;
    CE = 100;


    nlt = [13660    12200   10800   9480    8230    7090    6030    5080    4230    3490];
    I = size(nlt,2);

    J = 75;
    # PC = zeros(J,T);
    # PD = zeros(J,T);
    # EE = zeros(J,T+1);
    # DOD = zeros(J);
    # a = zeros(J,I);
    # FF = zeros(J);
    # NLT = zeros(J);
    # π = zeros(J);
    optimizer = Juniper.Optimizer
    nl_solver = optimizer_with_attributes(Ipopt.Optimizer, "print_level"=>0)
    m = Model(optimizer_with_attributes(optimizer, "nl_solver"=>nl_solver))

    @variable(m, Pc[t=1:T]);
    @variable(m, Pd[t=1:T]);
    @variable(m, E[t=1:T+1]);
    @variable(m, DoD);
    @variable(m, α[i=1:I], Bin);
    @variable(m, F);
    @variable(m, Nlt);

    @constraint(m, con1[t=1:T], 0 <= Pc[t] <= Pmax);
    @constraint(m, con2[t=1:T], -Pmax <= Pd[t] <= 0);
    @constraint(m, con3[t=1:T], E[t+1] == E[t] + (ηc*Pc[t]+Pd[t]/ηd)*Δt);
    @constraint(m, con4, E[1] == E[T+1]);
    @constraint(m, con5[t=1:T], 0 <= E[t] <= Emax);
    @constraint(m, con6, DoD == 1/(2*Emax)*sum(Pc[t]-Pd[t] for t=1:T)*Δt);
    @constraint(m, con7a, 0.1*sum((i-1)*α[i] for i=1:I) <= DoD);
    @constraint(m, con7b, DoD <= 0.1*sum(i*α[i] for i=1:I));
    @constraint(m, con8, sum(α[i] for i=1:I) == 1);
    @constraint(m, con9[t=tst:tfin], F <= -(Pc[t] + Pd[t]));
    @constraint(m, con10, Nlt == sum(α[i]*nlt[i] for i=1:I));

    @NLobjective(m, Max, F*π*(tfin-tst+1)*Δt - (sum((Pc[t] + Pd[t])*Cen for t=1:T)*Δt + Emax*CE/Nlt));

    optimize!(m);
    #JuMP.value.(Pc), JuMP.value.(Pd), JuMP.value.(E),
        return  JuMP.value.(Pc), JuMP.value.(Pd), JuMP.value.(E),JuMP.value.(F)
end
function hp_dsr(π)
        #hp inputs
    Δt = delt;
    ΔT = 11.5;
    tst = tb;
    tfin = te;
    # T = 48;
    Cen = 190;
    τamb1 = [-3 -3 -4 -4 -4 -4 -4 -4 -4 -4 -4 -4 -4 -4 -4 -4 -3 -3 -2 -2 -1 -1 0 0 2 2 2 2 2 2 3 3 3 3 3 3 3 3 4 4 4 4 4 4 5 5 5 5];
    τamb=τamb1[1:T];
    N = 4;
    #τtrg = 21;
    τmin = 20.5;
    τmax = 22;
    Pmax = ones(N)*3e-3;
    K = [160.3 111.4 76.4 38.1]/1e6;
    C = [15100/1.5 9800/1.5 7500/1.5 7400/1.5]/1e6;
    μ = [0.191 0.409 0.221 0.179];
    k_hp = 4;
    N_hp = Nhp;
    Cpen = 1;

        m = Model(optimizer_with_attributes(Ipopt.Optimizer, "tol" => 1e-6,"print_level" => 1))

        @variable(m, P[n=1:N,t=1:T]);
        @variable(m, τ[n=1:N,t=1:T+1]);
        @variable(m, F);
        @variable(m, τp[n=1:N,t=1:T]);
        @variable(m, τn[n=1:N,t=1:T]);

        @constraint(m, con1[n=1:N,t=1:T], 0 <= P[n,t] <= Pmax[n]);
        @constraint(m, con2[n=1:N,t=1:T], τ[n,t+1] == τ[n,t] - K[n]/C[n]*(τ[n,t]-τamb[t])*Δt + k_hp/C[n]*P[n,t]*Δt);
        @constraint(m, con3[n=1:N], τ[n,1] == τ[n,T+1]);
        @constraint(m, con4[n=1:N,t=1:T], 2*sum(P[n,r] for r=1:T) >= T*P[n,t]);
        @constraint(m, con5[t=tst:tfin], F <= N_hp*sum(μ[n]*(K[n]/k_hp*((τmin+τmax)/2-τamb[t]) - P[n,t]) for n=1:N));

        #@constraint(m, con6[n=1:N,t=1:T], τp[n,t] >= 0);
        @constraint(m, con7[n=1:N,t=1:T], τp[n,t] >= τ[n,t] - τmax);
        #@constraint(m, con8[n=1:N,t=1:T], τn[n,t] >= 0);
        @constraint(m, con9[n=1:N,t=1:T], τn[n,t] >= τmin - τ[n,t]);

        @objective(m, Max, F*π*(tfin-tst+1)*Δt - N_hp*sum(μ[n]*P[n,t]*Cen*Δt for n=1:N,t=1:T) - N_hp*Cpen/2*Δt*sum(μ[n]*(τn[n,t]^2 + τp[n,t]^2) for n=1:N,t=1:T))

        optimize!(m);

        return  JuMP.value.(F)
end
function ev_dsr(π)
        #inputs
    Δt = 0.5;
    ΔT = delt;
    # T = 48;
    tst = tb;
    tfin = te;
    Cen = 200;
    τav = 3.75;
    # Nev = 1000;

    Δe = 4*(-0.0188*τav + 1.2078)/1000;
    ΔE = Nev*Δe;
    Emax = Nev*25/1000;
    Pmax = ΔE/10;
    #Cars connected to charger - share
    CDF1 =    [0.998 0.999   1.000   1.000   0.999   0.999   0.998   0.997   0.994   0.991   0.975   0.959   0.904   0.849   0.739   0.628   0.557   0.487   0.436   0.385   0.326   0.266   0.204   0.142   0.071   0.001   0.230   0.235   0.24    0.245   0.275   0.30    0.38    0.45    0.56    0.67    0.77    0.87    0.89    0.91    0.922   0.943   0.957   0.972   0.98    0.988   0.992   0.997];
    Pdem1 = 0.001*Nev*[0.184 0.151   0.113   0.076   0.063   0.05    0.041   0.032   0.027   0.022   0.02    0.019   0.022   0.024   0.029   0.034   0.044   0.053   0.066   0.08    0.082   0.085   0.088   0.091   0.095   0.100   0.103   0.106   0.108   0.11    0.123   0.135   0.168   0.201   0.25    0.299   0.346   0.392   0.402   0.412   0.395   0.378   0.357   0.336   0.313   0.29    0.254   0.218];
    CDF=CDF1[1:T];
    Pdem=Pdem1[1:T];
    # END: Aggregator input data
    rb = 0.1;
    vb = 3.275;
    Cpen = 1000;

    m = Model(optimizer_with_attributes(Ipopt.Optimizer, "tol" => 1e-3,"print_level" => 1))

    @variable(m, Pb[t=1:T] >= 0);
    @variable(m, Pt[t=1:T] >= 0);
    @variable(m, E[t=1:T+1]);
    @variable(m, Ep[t=1:T] >= 0);
    @variable(m, F >= 0);

    # @constraint(m, con1a, E[27] == 0);
    @constraint(m, con1b[t=1:T], E[t+1] == E[t] + Pb[t]*Δt);
    # @constraint(m, con1c[t=27:T], E[t+1] == E[t] + Pb[t]*Δt);
    @constraint(m, con2, E[1] == E[T+1]);
    @constraint(m, con3[t=1:T], Pt[t] >= Pb[t] + rb/vb*Pb[t]^2);
    @constraint(m, con4[t=1:T], 0 <= Pt[t] <= Pmax*CDF[t]);
    @constraint(m, con5[t=tst:tfin], F <= Pdem[t] - Pt[t]);
    @constraint(m, con6[t=5:16], Ep[t] >= ΔE*(1 - CDF[t]) - E[t]);

    @objective(m, Max, F*π*(tfin-tst+1)*Δt - sum(Pt[t]*Cen*Δt for t=1:T) - Cpen/2*Δt*sum((Ep[t])^2 for t=1:T))

    optimize!(m);

    return JuMP.value.(Pb), JuMP.value.(Pt), JuMP.value.(E), JuMP.value.(Ep), JuMP.value.(F)
end
function Port_Supply_Curve_function(pi)
        N = 27; # Number of buses
        M = 26; # Number of branches
        S_base_new = 1;
        I_L_max = 4.1; # Maximum current of branch L
        SL_init = 15;

        ## Supply Curve
        P_orig = [3.1859, 0.4212, 2.8037, 1.6810, 2.8471, 1.0428, 1.5054, 0.1141, 1.7733, 1.2378, -0.0104, 1.7733, 1.9734, 2.1052, 3.2275, 4.5505, 0.9282, 0, 0, -1.247, 0, 0, 0.2189, -1.1227];
        T_FW = [tb te]; # Flexibility Window for example: 17:00 - 19:00
        DT_FW = T_FW[end] - T_FW[1] + 1; #Length of the Flexibility window



        # Initialization
        P = zeros(N,T);
        Q = zeros(N,T);

        ## Shore Power
        # Car carrier at TCT1; 1040 kW
        P[8,:] = [1.04 1.04 1.04 1.04 1.04 1.04 1.04 1.04 1.04 1.04 1.04 1.04 1.04 1.04 1.04 0 0 0 0 0 0 0 0 0]; # 00:00 - 15:00
        Q[8,:] = P[8,:]*0.20306;

        # Create combined load for:
        # Bulk carrier at TBT, 195 kW, [0 0 0 0 0 0 0.195 0.195 0.195 0.195 0.195 0.195 0.195 0.195 0.195 0.195 0.195 0 0 0 0 0 0 0]; # 06:00 - 17:00
        # Container ship at CT, 506 kW, [0.506 0.506 0.506 0.506 0.506 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]; # 00:00 - 05:00
        # Adding together shore power for TBT + CT
        P[5,:] = [0 0 0 0 0 0 0.195 0.195 0.195 0.195 0.195 0.195 0.195 0.195 0.195 0.195 0.195 0 0 0 0 0 0 0] + [0.506 0.506 0.506 0.506 0.506 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0];
        Q[5,:] = P[5,:]*0.20306;

        ## Energy Storage
        P_ES_max = 1.25;
        SOC_ES_max = 2.5;
        n_eff = 0.95;

        ## Industrial EVs
        # TBT; CT: binary variables which provide information whether terminals operate | not
        TBT = [0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0]; # 06:00 - 17:00
        CT = [1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]; # 00:00 - 05:00

        ##
        file = matopen("coalport.mat");
        mpc=read(file,"mpc");
        # mpc = Tyne_Coal_new;
        mpc["baseMVA"] = S_base_new; # New S_base
        mpc["branch"][:,3:4] = mpc["branch"][:,3:4]*(S_base_new/100); # New Z_base --> Z_pu' = Z_pu*(S_base_new/S_base_old)

        u_min = 0.94^2;
        u_max = 1.06^2;
        u_sb = 1.0; # voltage at bus 1 fixed
        sb = 1; # slack bus = 1

        ## Price & Carbon Intensity
        Price = [128.22 159.165 140.13 141.495 139.91 187.7 192.095 196.3 190.6 193.075 199.755 176.715 150.02 149.255 142.285 123.94 158.285 185.34 185.26 193.18 176.39 177.205 170.415 188.43]; # £/MWh, Fri 4 Feb 2022
        # Fri 4 Feb 2022; 87.92 GBP/tonne
        Carb_Int = 0.4*[68.5 77 74.5 71.5 75.5 86.5 104.5 137 145.5 142.5 143 129.5 113 111.5 106 114 133 147.5 147 148 142 129 105.5 79]; # gCO2/kWh | kgCO2/MWh [40# of 2022 level applied]
        Carb_Int = Carb_Int/1000; # --> tCO2/MWh
        Carb_Price = 87.92*Carb_Int

        ## Auxiliary Matrices [which branches are connected to each bus 1,...,n-1], 1 substation, bus sb
        # Power flow equations
        A4 = [1  [1   2   3   4   0   0   0]; # columns: 1) bus i, 2) lines [ls] connected to bus i [lines should be in accordance with buses js above], ALL BUSES [BUS 1 INCLUDED]
              2  [1   5   6   7   8   9   10];
              3  [2   16  0   0   0   0   0]
              4  [16  0   0   0   0   0   0]
              5  [3   13  14  15  0   0   0]
              6  [4   17  0   0   0   0   0]
              7  [17  0   0   0   0   0   0]
              8  [5   0   0   0   0   0   0]
              9  [6   18  0   0   0   0   0]
              10 [18  0   0   0   0   0   0]
              11 [7   19  0   0   0   0   0]
              12 [19  0   0   0   0   0   0]
              13 [8   20  0   0   0   0   0]
              14 [20  0   0   0   0   0   0]
              15 [9   21  0   0   0   0   0]
              16 [21  0   0   0   0   0   0]
              17 [10  11  12  0   0   0   0]
              18 [11  22  0   0   0   0   0]
              19 [22  0   0   0   0   0   0]
              20 [12  23  0   0   0   0   0]
              21 [23  0   0   0   0   0   0]
              22 [13  24  0   0   0   0   0]
              23 [24  0   0   0   0   0   0]
              24 [14  25  0   0   0   0   0]
              25 [25  0   0   0   0   0   0]
              26 [15  26  0   0   0   0   0]
              27 [26  0   0   0   0   0   0]];

       ##################################### defining the variables julia env

        # power flow variables u; L_ij; P_ij; Q_ij
        #Optimizer and its options
        optimizer = Juniper.Optimizer
        nl_solver = optimizer_with_attributes(Ipopt.Optimizer, "print_level"=>0);
        model = Model(optimizer_with_attributes(optimizer, "nl_solver"=>nl_solver));
        #######
        @variable(model, u[n=1:N,t=1:T]);
        @variable(model, L_ij[m=1:M,t=1:T]);
        @variable(model, Q_ij[m=1:M,t=1:T]);
        @variable(model, P_ij[m=1:M,t=1:T]);
        @variable(model, SL[t=1:T+1]);
        @variable(model, c1[t=1:T], Bin);
        @variable(model, c2[t=1:T], Bin);
        # SL =  # Ship Load
        # c1 =  # Crane 1
        # c2 =  # Crane 2
        # Industrial EV variables [Container Terminal]
        @variable(model, SOC_EH[t=1:T+1]);
        @variable(model, SOC_RS[t=1:T+1]);
        @variable(model, SOC_CT[t=1:T+1]);
        @variable(model, P_ch_EH[t=1:T]);
        @variable(model, P_ch_RS[t=1:T]);
        @variable(model, P_ch_CT[t=1:T]);
        P_dch_EH = 1*0.1*CT; # given
        P_dch_RS = 1*0.1*CT; # given
        P_dch_CT = 3*0.1*CT; # given
        # SOC_EH =  # Empty Handler
        # SOC_RS =  # Reach Stacker
        # SOC_CT = # Container Tractor
        # Industrial EV variables [Tyne Bulk Terminal]
        @variable(model, SOC_SL[t=1:T+1]);
        @variable(model, P_ch_SL[t=1:T]);
        P_dch_SL = 1*0.1*TBT; # given
        # SOC_SL =  # Shovel Loader

        # ESS variables
        @variable(model, SOC_ES[t=1:T+1]);
        @variable(model, P_ch_ES[t=1:T]);
        @variable(model, P_dch_ES[t=1:T]);
        @variable(model, beta[t=1:T], Bin);

        # Supply Curve variables
        @variable(model, P_sch[t=1:T]);
        @variable(model, P_F);

        # Define bounds
        @constraint(model, con1[n=1:N,t=1:T], u_min <= u[n,t] <= u_max);
        @constraint(model, con2[t=1:T], u[sb,:] .== u_sb);
        @constraint(model, con3[m=1:M,t=1:T], 0 <= L_ij[m,t] <= I_L_max^2);
        @constraint(model, con4[m=1:M,t=1:T], -10 <= P_ij[m,t] <= 10);
        @constraint(model, con5[m=1:M,t=1:T], -10 <= Q_ij[m,t] <= 10);
        @constraint(model, con6[t=1:T+1], -1 <= SL[t] <=  SL_init);
        @constraint(model, con6a1[t=1:6],  c1[t] ==  0);
        @constraint(model, con6a2[t=18:24],  c1[t] ==  0);
        @constraint(model, con6b1[t=1:6],  c2[t] ==  0);
        @constraint(model, con6b2[t=18:24],  c2[t] ==  0);
        @constraint(model, con7[t=1:T+1], 0 <= SOC_EH[t] <=  2*0.4);
        @constraint(model, con8[t=1:T+1], 0 <= SOC_RS[t] <=  3*0.4);
        @constraint(model, con9[t=1:T+1], 0 <= SOC_CT[t] <=  12*0.4);
        @constraint(model, con10[t=1:T+1], 0 <= SOC_SL[t] <=  2*0.4);
        @constraint(model, con11[t=1:T+1], 0 <= SOC_ES[t] <=  SOC_ES_max);
        @constraint(model, con12[t=1:T], 0 <= P_ch_EH[t]<=0.2+ (1 - CT[t])*0.2 );
        @constraint(model, con13[t=1:T], 0 <= P_ch_RS[t]<=0.4 + (1 - CT[t])*0.2  );
        @constraint(model, con14[t=1:T], 0 <= P_ch_CT[t] <= 1.8 + (1 - CT[t])*0.6);
        @constraint(model, con15[t=1:T], 0 <= P_ch_SL[t] <= 0.2 + (1 - TBT[t])*0.2);
        @constraint(model, con16[t=1:T], 0 <= P_ch_ES[t] <=P_ES_max );
        @constraint(model, con17[t=1:T], 0 <= P_dch_ES[t] <= P_ES_max);

        #Other types of Constraints
        cons_1 = []
        for t = 1:T
            for L = 1:M
                i = Int.(mpc["branch"][L,1]);
                @constraint(model,P_ij[L,t]^2 + Q_ij[L,t]^2 <= u[i,t]*L_ij[L,t]);
            end
        end

        # Formulation of branch flows
        for t = 1:T
            for L = 1:M
                j = Int.(mpc["branch"][L,2]);
                Ls1 = filter(x->x!=0,A4[j,2:end])'; # get branches L_k to which bus j is connected
                Ls= filter(x->x!=L,Ls1); # remove branch L[i-j] from Ls
                sigma_P_jk = 0;
                sigma_Q_jk = 0;
                for k2 = 1:length(Ls)
                  L_k = Ls[k2]; # get each branch L_k
                  sigma_P_jk = sigma_P_jk + P_ij[L_k,t];
                  sigma_Q_jk = sigma_Q_jk + Q_ij[L_k,t];
                end
                # Demand
                if (j .== 12) # Crane 1 + Hopper at bus 12
                    P_g_c = - 0.2655*c1[t];
                    Q_g_c = 0.20306*P_g_c;
                elseif j .== 14 # Crane 2 + Hopper at bus 14
                    P_g_c = - 0.2655*c2[t]
                    Q_g_c = 0.20306*P_g_c;
                elseif j .== 5 # Industrial EVs at CT and TBT [bus 5] & ESS [1.25 MW / 2.5 MWh]
                    P_g_c = - P[j,t] - P_ch_EH[t] - P_ch_RS[t] - P_ch_CT[t] - P_ch_SL[t] - P_ch_ES[t] + P_dch_ES[t]; # considering existing demand at bus 5
                    Q_g_c = 0.20306*P_g_c;
                else()
                    P_g_c = - P[j,t]; # demand only
                    Q_g_c = - Q[j,t];
                end
                @constraint(model,  P_ij[L,t] - sigma_P_jk - mpc["branch"][L,3]*L_ij[L,t] + P_g_c .== 0);
                @constraint(model,  Q_ij[L,t] - sigma_Q_jk - mpc["branch"][L,4]*L_ij[L,t] + Q_g_c .== 0);
            end
        end

        # u[j,t] .== u[i,t] - 2*(R[L]*P_ij[L,t] + X[L]*Q_ij[L,t]) + (R[L]^2 + X[L]^2)*L_ij[L,t] (Equation 20)
        # cons_4 = []
        for t = 1:T
            for L = 1:M
                i = Int.(mpc["branch"][L,1]);
                j = Int.(mpc["branch"][L,2]);
                @constraint(model, u[j,t] .== u[i,t] - 2*(mpc["branch"][L,3]*P_ij[L,t] + mpc["branch"][L,4]*Q_ij[L,t]) + (mpc["branch"][L,3]^2 + mpc["branch"][L,4]^2)*L_ij[L,t]);
            end
        end

        # Crane Constraints
        for t = 1:T
            @constraint(model, SL[t+1] == SL[t] - c1[t] - c2[t]);
            @constraint(model,SL[1] == SL_init);
            @constraint(model,SL[T+1] .== 0);
        end

        # Industrial EV Constraints
        @constraint(model, SOC_EH[1] .== 2*0.4);
        @constraint(model,SOC_EH[T+1] .== 2*0.4);
        @constraint(model, SOC_EH[1] .== 2*0.4);
        @constraint(model,  SOC_RS[T+1] .== 3*0.4);
        @constraint(model, SOC_SL[1] == 2*0.4);
        @constraint(model, SOC_SL[T+1] .== 2*0.4);
        @constraint(model,SOC_CT[T+1] .== 12*0.4);

        for t = 1:T
            @constraint(model, SOC_EH[t+1] == SOC_EH[t] + P_ch_EH[t]*n_eff - P_dch_EH[t]/n_eff ) ;
            @constraint(model, SOC_RS[t+1] == SOC_RS[t] + P_ch_RS[t]*n_eff - P_dch_EH[t]/n_eff);
            @constraint(model,SOC_CT[t+1] == SOC_CT[t] + P_ch_CT[t]*n_eff - P_dch_CT[t]/n_eff);
            @constraint(model, SOC_SL[t+1] == SOC_SL[t] + P_ch_SL[t]*n_eff - P_dch_SL[t]/n_eff);
            @constraint(model,SOC_RS[1] == 3*0.4);
            @constraint(model, SOC_CT[1] == 12*0.4);

        end

        # ESS constraints [1]
        @constraint(model,SOC_ES[1] .== SOC_ES[T+1]);
        for t = 1:T
            @constraint(model, SOC_ES[t+1] == SOC_ES[t] + P_ch_ES[t]*n_eff - P_dch_ES[t]/n_eff );
        end

        # ESS constraints [2]
        @constraint(model,P_ch_ES .<= P_ES_max*beta);
        @constraint(model,P_dch_ES .<= P_ES_max*(ones(length(beta)) - beta));

        # P_sch[t] = P_ij[1,t] + P_ij[2,t] + P_ij[3,t] + P_ij[4,t]
        for t = 1:T
            @constraint(model, P_sch[t] .== sum(P_ij[1:4,t]));
        end

        # P_F <= P_orig[t] - P_sch[t],
        for t = T_FW[1]:T_FW[end]
            @constraint(model, P_F <= P_orig[t] - P_sch[t]); # P_orig[t] >= P_sch[t]
        end

        @objective(model, Max, pi*P_F*DT_FW - sum( P_sch[t]*(Price[t] + Carb_Price[t]) for t=1:T) )
        optimize!(model);

        return JuMP.value.(P_F)
end

function buildcurves(tb,te,T,delt,Pmaxes,Nev,Nhp)

####ES##########################################################################

PCes = zeros(I,T);
PDes = zeros(I,T);
EEes = zeros(I,T+1);
FFes = zeros(I);
FFes_m = zeros(I);
pres = zeros(I);
fb=0;
for i=1:I
    pres[i] = i*3/1000;
    (PCes[i,:],PDes[i,:],EEes[i,:],FFes[i]) = es_dsr(pres[i]);
    FFes_m[i]=FFes[i]-fb;
    fb=FFes[i];
end
#hp#########################
FFhp_m = zeros(I);
FFhp = zeros(I);
prhp = zeros(I);
fb=0;
for i=1:I
    prhp[i] = i*6;
    (FFhp[i]) = hp_dsr(prhp[i]);
    FFhp_m[i]=FFhp[i]-fb;
    fb=FFhp[i];
end
########################################################################
#EV#####################################################################

PBev = zeros(I,T);
PTev = zeros(I,T);
EEev = zeros(I,T+1);
EPev = zeros(I,T);
FFev = zeros(I);
FFev_m = zeros(I);
prev = zeros(I);
fb=0;
for i=1:I
    prev[i] = i*8;
    (PBev[i,:],PTev[i,:],EEev[i,:],EPev[i,:],FFev[i]) = ev_dsr(prev[i]);
    FFev_m[i]=FFev[i]-fb;
    fb=FFev[i];
end
##############################################################################################
#IC
Δt =delt;
ΔT = te-tb+1;
T = Int((ΔT + 2*ΔT)/Δt);
Cen = 200*ones(T);
Pmax = 0.928;
A = 17.65;
B = 11.76;
a = A/Pmax;
b = B*2;
G = I;
pric=  zeros(G);
FFic = zeros(G);
FFic_m=zeros(G);
PPic = zeros(G,T);
fb=0;
for g=1:G

    pric[g] = g*8;

    m = Model(optimizer_with_attributes(Ipopt.Optimizer, "tol" => 1e-6,"print_level" => 1))

    @variable(m, 0 <= P[t=1:T] <= Pmax);
    @variable(m, 0 <= F <= Pmax);

    @constraint(m, con1[t=1:Int(ΔT/Δt)], F == Pmax - P[t]);
    @constraint(m, con2[t=Int(ΔT/Δt+1):T], P[t] <= F/2);
    @constraint(m, con3, F*ΔT == sum(P[t]*Δt for t=Int(ΔT/Δt+1):T));

    @objective(m, Max, F*pric[g]*ΔT - (a*F^2 + b*F) - sum(Cen[t]*P[t]*Δt for t=1:T))

    optimize!(m);

    FFic[g] = JuMP.value.(F);
    PPic[g,:] = JuMP.value.(P);
    FFic_m[g]=FFic[g]-fb;
    fb=FFic[g];
end
##################################################
#port
    fb=0;
    FFport=zeros(I);
    pscport=zeros(I,T);
    for i=1:I
        prport=i*20;
        (FFport[i]) = Port_Supply_Curve_function(prport);
        FFport_m[i]=FFport[i]-fb;
        fb=FFport[i];
    end
    return JuMP.value.(FFhp), JuMP.value.(FFev), JuMP.value.(FFes), JuMP.value.(FFic),JuMP.value.(FFport),JuMP.value.(FFhp_m), JuMP.value.(FFev_m), JuMP.value.(FFes_m), JuMP.value.(FFic_m),JuMP.value.(FFport_m), JuMP.value.(prhp),JuMP.value.(prev), JuMP.value.(pres), JuMP.value.(pric), JuMP.value.(prport)
end
I=5;
tb=19;
te=21;
T=24;
delt=1;
Pmaxes=1000;
Nev=1000;
Nhp=1000;
FFhp=zeros(I);
FFev=zeros(I);
FFes=zeros(I);
FFic=zeros(I);
FFport=zeros(I);
FFhp_m=zeros(I);
FFev_m=zeros(I);
FFes_m=zeros(I);
FFic_m=zeros(I);
FFport_m=zeros(I);
prhp=zeros(I);
prev=zeros(I);
pres=zeros(I);
pric=zeros(I);
prport=zeros(I);

(FFhp,FFev,FFes, FFic,FFport,FFhp_m,FFev_m,FFes_m, FFic_m,FFport_m,prhp,prev,pres,pric, prport)=buildcurves(tb,te,T,delt,Pmaxes,Nev,Nhp);

#save data for the later use
open("FF.txt", "w") do io
    writedlm(io, [FFhp FFev FFes FFic FFport])
end

open("FFm.txt", "w") do io
    writedlm(io, [FFhp_m FFev_m FFes_m  FFic_m FFport_m])
end
open("price.txt", "w") do io
    writedlm(io, [prhp prev pres pric prport])
end
# jj=readdlm("data.txt", '\t', Float64, '\n')



#branches: 3
Obj: -13.317884927987686
nl_solver         : MathOptInterface.OptimizerWithAttributes(Ipopt.Optimizer, Pair{MathOptInterface.AbstractOptimizerAttribute, Any}[MathOptInterface.RawOptimizerAttribute("print_level") => 0])
feasibility_pump  : false
log_levels        : [:Options, :Table, :Info]

#Variables: 2911
#IntBinVar: 72
Obj Sense: Max

Start values are not feasible.
Status of relaxation: LOCALLY_SOLVED
Time for relaxation: 14.047413110733032
Relaxation Obj: -4367.683377185397

 ONodes   CLevel          Incumbent                   BestBound            Gap    Time   Restarts  GainGap  
    2       2                 -                        -4367.68             -    103.3      0         -     


[ Info: Breaking out of strong branching as the time limit of 100.0 seconds got reached.


    3       3                 -                        -4367.68             -    109.2      -       55.9%   
    4       4                 -                        -4367.68             -    116.1      -       11.8%   
    5       5                 -                        -4367.68             -    121.5      -        0.6%   
    6       6                 -                        -4367.69             -    128.2      -       30.9%   
    7       7                 -                        -4367.73             -    133.5      -       11.8%   
    8       8                 -                        -4367.73             -    140.3      -       50.6%   
    9       9                 -                        -4367.8              -    145.5      -       16.7%   
   10       10                -                        -4367.8              -    151.4      -       16.7%   
   11       11                -                        -4367.8              -    157.7      -       100.0%  
   12       12     

[ Info: Breaking out of strong branching as the time limit of 100.0 seconds got reached.


    3       3                 -                        -4367.67             -    107.7      -       55.9%   
    4       4                 -                        -4367.67             -    113.1      -       11.8%   
    5       5                 -                        -4367.67             -    119.2      -        0.6%   
    6       6                 -                        -4367.68             -    124.7      -       30.9%   
    7       7                 -                        -4367.71             -    131.4      -       11.8%   
    8       8                 -                        -4367.71             -    136.7      -       50.6%   
    9       9                 -                        -4367.79             -    142.7      -       16.7%   
   10       10                -                        -4367.79             -    148.3      -       16.7%   
   11       11                -                        -4367.79             -    153.9      -       100.0%  
   12       12     

...

    3       3                 -                        -4367.66             -    107.8      -       55.9%   
    4       4                 -                        -4367.66             -    114.3      -       11.8%   
    5       5                 -                        -4367.66             -    119.4      -        0.6%   
    6       6                 -                        -4367.67             -    125.9      -       30.9%   
    7       7                 -                        -4367.7              -    131.1      -       11.8%   
    8       8                 -                        -4367.7              -    137.6      -       50.5%   
    9       9                 -                        -4367.77             -    143.0      -       16.7%   
   10       10                -                        -4367.77             -    148.7      -       16.7%   
   11       11                -                        -4367.77             -    155.3      -       100.0%  
   12       12     

[ Info: Breaking out of strong branching as the time limit of 100.0 seconds got reached.


    3       3                 -                        -4367.64             -    107.5      -       55.9%   
    4       4                 -                        -4367.64             -    113.1      -       11.8%   
    5       5                 -                        -4367.64             -    120.0      -        0.6%   
    6       6                 -                        -4367.66             -    125.4      -       30.9%   
    7       7                 -                        -4367.69             -    131.9      -       11.8%   
    8       8                 -                        -4367.69             -    137.1      -       50.6%   
    9       9                 -                        -4367.76             -    144.0      -       16.7%   
   10       10                -                        -4367.76             -    149.8      -       100.0%  
   11       11                -                        -4367.76             -    156.6      -       100.0%  
   12       12     

[ Info: Breaking out of strong branching as the time limit of 100.0 seconds got reached.


    3       3                 -                        -4367.63             -    110.5      -       55.9%   
    4       4                 -                        -4367.63             -    116.2      -       11.8%   
    5       5                 -                        -4367.63             -    122.9      -        0.6%   
    6       6                 -                        -4367.64             -    128.7      -       30.9%   
    7       7                 -                        -4367.68             -    135.7      -       11.8%   
    8       8                 -                        -4367.68             -    141.2      -       50.6%   
    9       9                 -                        -4367.75             -    148.1      -       16.7%   
   10       10                -                        -4367.75             -    154.0      -       100.0%  
   11       11                -                        -4367.75             -    160.8      -       100.0%  
   12       12     